In [1]:
import polars as pl

# Load lazily to reduce RAM usage
df = pl.read_parquet("../data/malaysia_transactions.parquet")

In [2]:
columns_to_keep = [
    "date_time",
    "ofi_entity_id",
    "rfi_entity_id",
    "trxn_amount",
    "trxn_type",
    "trxn_channel"
]

df_clean = df.select(columns_to_keep)

df_clean.null_count()

date_time,ofi_entity_id,rfi_entity_id,trxn_amount,trxn_type,trxn_channel
u32,u32,u32,u32,u32,u32
0,438281,464000,0,665434,0


In [3]:
# Drop any rows with missing sender/receiver IDs
df_filtered = df_clean.filter(
    pl.col("ofi_entity_id").is_not_null() & 
    pl.col("rfi_entity_id").is_not_null()
)

# Fill missing `trxn_type` with fallback label
df_filtered = df_filtered.with_columns(
    pl.col("trxn_type").fill_null("Unknown")
)

In [4]:
df_filtered.null_count()

date_time,ofi_entity_id,rfi_entity_id,trxn_amount,trxn_type,trxn_channel
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


In [5]:
df_filtered.shape

(11396551, 6)

In [6]:
# Sort the dataframe by time
df_sorted = df_filtered.sort("date_time")

# check earliest and latest timestamp
print("Start:", df_sorted["date_time"][0])
print("End:", df_sorted["date_time"][-1])

Start: 2025-06-01 00:00:00
End: 2025-06-30 23:59:59


In [6]:
from collections import defaultdict
from datetime import datetime, timedelta

# Parameters
MAX_LAYERING_DEPTH = 4
MAX_TIME_GAP_SECONDS = 600  # 10 minutes
FANOUT_THRESHOLD = 0.7
DECAY_FACTOR = 0.9
BASE_TRANSACTION_RISK = 0.02
INITIAL_FANIN_THRESHOLD = 3
INITIAL_FANIN_RISK = 0.5
FANOUT_BOOST = 0.3
QUICK_FANOUT_TIME = timedelta(hours=1)

# Internal state
graph = defaultdict(list)
incoming_graph = defaultdict(list)
last_incoming_time = {}
last_outgoing_time = {}
entity_risk = defaultdict(float)
entity_balance = defaultdict(float)
fanin_count = defaultdict(int)
fanin_today = defaultdict(list)
fanout_today = defaultdict(list)
first_fanin_time = {}

def ensure_datetime(ts):
    return ts if isinstance(ts, datetime) else datetime.strptime(str(ts), "%Y-%m-%d %H:%M:%S")

def is_new_account(entity):
    return fanin_count[entity] < 5

def update_balance(sender, receiver, amount):
    entity_balance[sender] -= amount
    entity_balance[receiver] += amount

def decay_risk(entity):
    entity_risk[entity] *= DECAY_FACTOR

def process_transaction(sender, receiver, timestamp, amount, ttype):
    timestamp = ensure_datetime(timestamp)
    today = timestamp.date()

    # Track structure
    graph[sender].append((receiver, timestamp, amount, ttype))
    incoming_graph[receiver].append((sender, timestamp, amount, ttype))
    update_balance(sender, receiver, amount)

    # Base risk for every transaction
    entity_risk[sender] += BASE_TRANSACTION_RISK
    entity_risk[receiver] += BASE_TRANSACTION_RISK

    # Fan-in tracking
    fanin_today[(receiver, today)].append((sender, amount, timestamp))
    fanin_count[receiver] += 1

    if receiver not in first_fanin_time:
        first_fanin_time[receiver] = timestamp

    # Fan-in burst detection
    if is_new_account(receiver) and len(fanin_today[(receiver, today)]) >= INITIAL_FANIN_THRESHOLD:
        time_window = timestamp - first_fanin_time[receiver]
        if time_window < timedelta(hours=1):
            entity_risk[receiver] = max(entity_risk[receiver], INITIAL_FANIN_RISK)

    # Fan-out tracking
    fanout_today[(sender, today)].append((receiver, amount, timestamp))

    # Fan-out risk if quick drain after fan-in
    outgoing = sum(a for _, a, _ in fanout_today[(sender, today)])
    balance = outgoing + entity_balance[sender]
    if balance > 0:
        drain_ratio = outgoing / balance
        if drain_ratio > FANOUT_THRESHOLD:
            # Check time since first fan-in
            if sender in first_fanin_time:
                if (timestamp - first_fanin_time[sender]) <= QUICK_FANOUT_TIME:
                    entity_risk[sender] += FANOUT_BOOST

    # Decay risks
    decay_risk(sender)
    decay_risk(receiver)

    return max(entity_risk[sender], entity_risk[receiver])

def layering_depth(entity, visited, current_time, depth=0):
    if depth >= MAX_LAYERING_DEPTH:
        return depth

    max_depth = depth
    for (target, t, amt, ttype) in graph[entity]:
        t = ensure_datetime(t)
        if target not in visited and abs((current_time - t).total_seconds()) <= MAX_TIME_GAP_SECONDS:
            visited.add(target)
            new_depth = layering_depth(target, visited, t, depth + 1)
            max_depth = max(max_depth, new_depth)
            visited.remove(target)

    return max_depth

def layering_depth_backtrace(entity, visited, current_time, depth=0):
    if depth >= MAX_LAYERING_DEPTH:
        return depth

    max_depth = depth
    for (source, t, amt, ttype) in incoming_graph[entity]:
        t = ensure_datetime(t)
        if source not in visited and abs((current_time - t).total_seconds()) <= MAX_TIME_GAP_SECONDS:
            visited.add(source)
            new_depth = layering_depth_backtrace(source, visited, t, depth + 1)
            max_depth = max(max_depth, new_depth)
            visited.remove(source)

    return max_depth

In [ ]:
# Reset state
graph.clear()
incoming_graph.clear()
last_incoming_time.clear()
last_outgoing_time.clear()
entity_risk.clear()
entity_balance.clear()
fanin_count.clear()
fanin_today.clear()
fanout_today.clear()
first_fanin_time.clear()

# Time setup
now = datetime.now()
later = now + timedelta(minutes=5)
later2 = now + timedelta(minutes=10)
later3 = now + timedelta(minutes=15)

results = {}

# Test 1: Group meal (low risk)
results['group_meal'] = [
    process_transaction("A1", "X", now, 12.5, "p2p"),
    process_transaction("A2", "X", later, 13.0, "p2p"),
    process_transaction("A3", "X", later2, 11.8, "p2p"),
]
print("Group meal risk:", results["group_meal"])

# Test 2: Mule-style burst fan-in
results['mule_fanin'] = [
    process_transaction("B1", "Y", now, 3000.0, "p2p"),
    process_transaction("B2", "Y", later, 2999.0, "p2p"),
    process_transaction("B3", "Y", later2, 3100.0, "p2p"),
]
print("Mule fan-in risk:", results["mule_fanin"])

# Test 3: Fan-out after burst
results['mule_fanout'] = [
    process_transaction("Y", "C1", later2 + timedelta(minutes=1), 2000.0, "transfer"),
    process_transaction("Y", "C2", later2 + timedelta(minutes=2), 2500.0, "transfer"),
    process_transaction("Y", "C3", later2 + timedelta(minutes=3), 2500.0, "transfer"),
]
print("Mule fan-out risk:", results["mule_fanout"])

# Test 4: Legit business spaced out over time
t1 = now.replace(hour=10, minute=0)
t2 = now.replace(hour=11, minute=0)
t3 = now.replace(hour=12, minute=0)
t4 = now.replace(hour=13, minute=0)
t5 = now.replace(hour=14, minute=0)

results['legit_business'] = [
    process_transaction("D1", "Z", t1, 105.00, "p2b"),
    process_transaction("D2", "Z", t2, 95.00, "p2b"),
    process_transaction("D3", "Z", t3, 110.00, "p2b"),
    process_transaction("D4", "Z", t4, 90.00, "p2b"),
    process_transaction("D5", "Z", t5, 100.00, "p2b"),
]
print("Legit business risk:", results["legit_business"])

# Final entity risk scores
print({
    "X (group meal)": entity_risk["X"],
    "Y (mule)": entity_risk["Y"],
    "Z (business)": entity_risk["Z"]
})

{'X (group meal)': 0.45, 'Y (mule)': 0.6468300000000001, 'Z (business)': 0.07371180000000001}


In [15]:
# from datetime import datetime, timedelta

# base_time = datetime(2025, 6, 1, 12, 0, 0)

# for i in range(10):
#     s = f"E{i:03}"
#     r = f"E{i+1:03}"
#     t = base_time + timedelta(minutes=i)  # try shorter gap for first few
#     a = 1000.0 - i * 10
#     process_transaction(s, r, t, a, "Online Transfer")

⚠️ Transaction blocked: Risk too high for E002 (0.96)
⚠️ Transaction blocked: Risk too high for E003 (0.96)
⚠️ Transaction blocked: Risk too high for E004 (0.96)
⚠️ Transaction blocked: Risk too high for E005 (0.96)
⚠️ Transaction blocked: Risk too high for E006 (0.96)
⚠️ Transaction blocked: Risk too high for E007 (0.96)
⚠️ Transaction blocked: Risk too high for E008 (0.96)
⚠️ Transaction blocked: Risk too high for E009 (0.96)
⚠️ Transaction blocked: Risk too high for E010 (0.96)


In [14]:
# print("Risk on E002:", entity_risk["E002"])
# print("Risk on E003:", entity_risk["E003"])
# print("Risk on E004:", entity_risk["E004"])
# print("Risk on E005:", entity_risk["E005"])

Risk on E002: 0.8
Risk on E003: 0.8
Risk on E004: 0.8
Risk on E005: 0.8


In [2]:
# from datetime import datetime, timedelta

# base_time = datetime(2025, 6, 1, 12, 0, 0)

# synthetic_chain = [
#     ("E001", "E002", base_time, 1000.0, "Online Transfer"),
#     ("E002", "E003", base_time + timedelta(minutes=1), 980.0, "Online Transfer"),
#     ("E003", "E004", base_time + timedelta(minutes=2), 970.0, "Online Transfer"),
#     ("E004", "E005", base_time + timedelta(minutes=3), 950.0, "Online Transfer"),
# ]

# for s, r, t, a, tt in synthetic_chain:
#     process_transaction(s, r, t, a, tt)

# print("Risk on E002:", entity_risk["E002"])
# print("Risk on E003:", entity_risk["E003"])
# print("Risk on E004:", entity_risk["E004"])
# print("Risk on E005:", entity_risk["E005"])

NameError: name 'process_transaction' is not defined

In [9]:
# row = df_sorted.row(0)
# process_transaction(
#     sender=row[1],          # ofi_entity_id
#     receiver=row[2],        # rfi_entity_id
#     timestamp=row[0],       # date_time
#     amount=row[3],          # trxn_amount
#     ttype=row[4]            # trxn_type
# )

In [10]:
# N = 100000  # start small, increase later if fast

# for i in range(N):
#     row = df_sorted.row(i)
#     process_transaction(
#         sender=row[1],
#         receiver=row[2],
#         timestamp=row[0],
#         amount=row[3],
#         ttype=row[4]
#     )

# # Inspect: how many entities have non-zero risk
# non_zero_risk = {k: v for k, v in entity_risk.items() if v > 0}
# print("Entities with non-zero risk:", len(non_zero_risk))

# # Show top 10 riskiest entities
# top_risky = sorted(non_zero_risk.items(), key=lambda x: x[1], reverse=True)[:10]
# print("Top risky entities:", top_risky)

Entities with non-zero risk: 0
Top risky entities: []
